# Sesion 6 — T4 Cierre: Modulos Propios + T5 Inicio: Pandas
## Diplomado: Machine Learning en Seguros
### Facultad de Ciencias, UNAM · 25 de abril de 2026  (07:00 - 11:00 h  — 4 horas)

---

**Instructor:** Eric   **Sesion:** 6 de 13

---

> **Objetivos:**
> 1. Cerrar T4: crear modulos propios, usar tabla_tarifas.py y scripts .py
> 2. Abrir T5: entender Series y DataFrames de Pandas desde cero

## Contenido

**T4 Cierre — Modulos propios**
1. [Crear mi_modulo.py desde cero](#1-modulo)
2. [Importar y usar el modulo en el notebook](#2-importar)
3. [tabla_tarifas.py — clase actuarial real](#3-tarifas)
4. [Scripts .py — if __name__ == '__main__'](#4-scripts)

**T5 Inicio — Pandas**
5. [Que es Pandas y por que usarlo](#5-pandas)
6. [Series — el array etiquetado](#6-series)
7. [DataFrame — la tabla de Pandas](#7-dataframe)
8. [Acceder, filtrar y crear columnas](#8-acceso)
9. [Ejercicio integrador](#9-ejercicio)

---
<a id='1-modulo'></a>
## 1. Crear mi_modulo.py desde Cero

Un modulo propio es simplemente un archivo `.py` con funciones.
Se guarda en la misma carpeta que el notebook para poder importarlo.

> **Accion:** Abre VS Code → Archivo → Nuevo → guarda como `mi_modulo.py`
> en la misma carpeta que este notebook.

**Contenido de mi_modulo.py** — escribe esto en VS Code:

In [1]:
# Este bloque muestra el contenido que debes escribir en mi_modulo.py
# NO lo ejecutes aqui — es para que lo copies al archivo .py

contenido_mi_modulo = '''
"""
Modulo de funciones actuariales para el Diplomado ML en Seguros.
Facultad de Ciencias, UNAM — Modulo 1.
"""

# ── Funciones de primas ──────────────────────────────────────────────────
def calcular_prima(suma_asegurada, tasa_riesgo, recargo=0.15, iva=1.16):
    """Calcula la prima comercial de una poliza de seguros."""
    prima_pura = suma_asegurada * tasa_riesgo
    return prima_pura * (1 + recargo) * iva

def prima_mensual(suma_asegurada, tasa_riesgo, **kwargs):
    """Devuelve la prima mensual dividiendo entre 12."""
    return calcular_prima(suma_asegurada, tasa_riesgo, **kwargs) / 12

# ── Funciones de clasificacion ───────────────────────────────────────────
def clasificar_riesgo(siniestros):
    """Clasifica nivel de riesgo por numero de siniestros.
    Retorna: 'ALTO', 'MEDIO' o 'BAJO'
    """
    if siniestros >= 3:   return 'ALTO'
    elif siniestros >= 1: return 'MEDIO'
    return 'BAJO'

def grupo_edad(edad):
    """Devuelve el grupo tarifario por edad."""
    if edad <= 30:   return '18-30'
    elif edad <= 45: return '31-45'
    elif edad <= 60: return '46-60'
    return '61+'

# ── Metricas de cartera ───────────────────────────────────────────────────
def metricas_cartera(*primas):
    """Calcula metricas basicas de una cartera.
    Retorna dict con n, total, promedio, maximo, minimo.
    """
    if not primas: return None
    return dict(
        n=len(primas), total=sum(primas),
        promedio=sum(primas)/len(primas),
        maximo=max(primas), minimo=min(primas)
    )
'''

print('Contenido mostrado — ahora copialo a mi_modulo.py en VS Code')
print(f'{len(contenido_mi_modulo.splitlines())} lineas')

Contenido mostrado — ahora copialo a mi_modulo.py en VS Code
43 lineas


---
<a id='2-importar'></a>
### 2. Importar y Usar el Modulo en el Notebook

Una vez guardado `mi_modulo.py` en la misma carpeta, importalo aqui:

In [1]:
# Importar el modulo completo
import mi_modulo

# Ver que contiene
print(dir(mi_modulo))
print()

# Usar sus funciones con el prefijo mi_modulo.
p = mi_modulo.calcular_prima(500_000, 0.022)
print(f'Prima: ${p:,.2f}')

r = mi_modulo.clasificar_riesgo(2)
print(f'Riesgo: {r}')

g = mi_modulo.grupo_edad(47)
print(f'Grupo: {g}')

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calcular_prima', 'clasificar_riesgo', 'grupo_edad', 'hola', 'metricas_cartera', 'prima_mensual']

Prima: $14,674.00
Riesgo: MEDIO
Grupo: 46-60


In [2]:
# Importar solo las funciones que necesitas (sin prefijo)
from mi_modulo import calcular_prima, clasificar_riesgo, grupo_edad, metricas_cartera

# Ahora se usan sin el prefijo
print(calcular_prima(1_000_000, 0.031))
print(clasificar_riesgo(0))
print(grupo_edad(55))

# Probar metricas_cartera con *args
res = metricas_cartera(2400, 6200, 1900, 14500, 3200)
for k, v in res.items():
    print(f'  {k:<10}: {v:>10,.2f}')

41354.0
BAJO
46-60
  n         :       5.00
  total     :  28,200.00
  promedio  :   5,640.00
  maximo    :  14,500.00
  minimo    :   1,900.00


In [22]:
# Si modificas mi_modulo.py y quieres recargar sin reiniciar el kernel:
import importlib
import mi_modulo
importlib.reload(mi_modulo)
print('Modulo recargado')

Modulo recargado


---
<a id='3-tarifas'></a>
### 3. tabla_tarifas.py — Clase Actuarial Real

`tabla_tarifas.py` ya existe en el repositorio del curso.
Contiene una **clase** (un tipo especial de modulo) que representa una tabla de tarifas de GMM.
Hoy la usamos como ejemplo de modulo real bien documentado.

> **Nota:** Las clases son un concepto avanzado que veremos en profundidad mas adelante.
> Por ahora solo aprende a usarla — no te preocupes por entender todo su codigo interno.

In [4]:
# Importar la clase Tablatarifas del modulo tabla_tarifas
from tabla_tarifas import Tablatarifas

# Crear una tabla de tarifas para polizas con historial COVID
# Parametros: edad_inicio, edad_final, brinco (salto), piso (True/False)
tarifa_covid = Tablatarifas(
    edad_inicio=45,
    edad_final=65,
    brinco=5,
    piso=False
)

# Ver que edades cubre la tabla
print('Edades:', tarifa_covid.edades)
print('Numero de franjas:', len(tarifa_covid.edades))

Edades: [45, 50, 55, 60, 65]
Numero de franjas: 5


In [5]:
# Asignar una tarifa a cada franja de edad
# Debe haber exactamente un valor por franja
exito = tarifa_covid.set_tarifa([5000, 6000, 8000, 10000, 12000])
print(f'Tarifa asignada: {exito}')

# Ver la tabla completa (el __repr__ del modulo)
print(tarifa_covid)

Tarifa asignada: True
edad 45 tiene una tarifa de 5000
edad 50 tiene una tarifa de 6000
edad 55 tiene una tarifa de 8000
edad 60 tiene una tarifa de 10000
edad 65 tiene una tarifa de 12000


In [7]:
# Consultar la tarifa para una edad especifica
print(tarifa_covid.get_tarifa(52))   # 8000
print(tarifa_covid.get_tarifa(45))   # 5000  (primer rango)
print(tarifa_covid.get_tarifa(70))   # 12000 (fuera de rango → max)
print(tarifa_covid.get_tarifa(30))   # 5000  (fuera de rango → min)

# Crear una segunda tabla — polizas sin COVID
tarifa_nocovid = Tablatarifas(45, 65, 5, True)
tarifa_nocovid.set_tarifa([2500, 3000, 4000, 5000, 6000])

# Comparar tarifas para la misma edad
edad_prueba = 52
print(f'Edad {edad_prueba}:')
print(f'  Con COVID:    ${tarifa_covid.get_tarifa(edad_prueba):,}')
print(f'  Sin COVID:    ${tarifa_nocovid.get_tarifa(edad_prueba):,}')

8000
5000
12000
5000
Edad 52:
  Con COVID:    $8,000
  Sin COVID:    $3,000


In [8]:
# Aplicar la tabla a una lista de asegurados
asegurados = [
    ('Ana Torres',  28),
    ('Luis Perez',  45),
    ('Maria Garcia',52),
    ('Carlos Ruiz', 62),
    ('Rosa Mendez', 38),
]

print(f'{"Asegurado":<18} {"Edad":>5} {"Tarifa COVID":>14} {"Tarifa Normal":>14}')
print('-' * 55)
for nombre, edad in asegurados:
    t_covid   = tarifa_covid.get_tarifa(edad)
    t_nocovid = tarifa_nocovid.get_tarifa(edad)
    print(f'{nombre:<18} {edad:>5} ${t_covid:>13,} ${t_nocovid:>13,}')

Asegurado           Edad   Tarifa COVID  Tarifa Normal
-------------------------------------------------------
Ana Torres            28 $        5,000 $        2,500
Luis Perez            45 $        5,000 $        2,500
Maria Garcia          52 $        8,000 $        3,000
Carlos Ruiz           62 $       12,000 $        5,000
Rosa Mendez           38 $        5,000 $        2,500


---
<a id='4-scripts'></a>
### 4. Scripts .py — if __name__ == '__main__'

Un script es un archivo `.py` que se ejecuta directamente desde la terminal.
El patron `if __name__ == '__main__'` permite que un archivo funcione
tanto como modulo importable como script ejecutable.

**Ejecutar desde Anaconda Prompt:**
```bash
cd 'C:\Users\TuNombre\Documents\diplomado-ml-seguros\modulo1\sesion6'
python mi_script.py
```

In [9]:
# Demostrar __name__ en el notebook
print('En el notebook, __name__ es:', __name__)
# __main__  ← porque es el archivo que se ejecuto directamente

# En un modulo importado, __name__ seria el nombre del archivo
import mi_modulo
print('En mi_modulo, __name__ es:', mi_modulo.__name__)
# mi_modulo  ← porque fue importado, no ejecutado directamente

En el notebook, __name__ es: __main__
En mi_modulo, __name__ es: mi_modulo


**Estructura de un script bien escrito:**

```python
# reporte_cartera.py
"""Script para generar reporte mensual de cartera."""
from mi_modulo import calcular_prima, clasificar_riesgo

def generar_reporte(cartera):
    """Genera el reporte."""
    for poliza in cartera:
        prima  = calcular_prima(poliza['suma'], poliza['tasa'])
        riesgo = clasificar_riesgo(poliza['siniest'])
        print(f"{poliza['id']}: ${prima:,.2f} [{riesgo}]")

# Este bloque SOLO se ejecuta si corres: python reporte_cartera.py
# NO se ejecuta si haces: import reporte_cartera
if __name__ == '__main__':
    datos = [
        {'id':'P01', 'suma':500_000, 'tasa':0.022, 'siniest':0},
        {'id':'P02', 'suma':1_000_000, 'tasa':0.031, 'siniest':2},
    ]
    generar_reporte(datos)
```

---
<a id='5-pandas'></a>
## T5 — Manejo de Datos con Pandas

### 5. Que es Pandas y Por Que Usarlo

**Pandas** es la biblioteca mas importante para analisis de datos en Python.
Es la base de todo lo que haremos en los Modulos 2, 3, 4 y 5.

| Operacion | Excel | Pandas |
|-----------|-------|--------|
| Cargar datos | Abrir archivo | `pd.read_csv()` |
| Filtrar filas | Autofiltro | `df[df['col'] > valor]` |
| Agrupar | Tabla dinamica | `df.groupby()` |
| Unir tablas | BUSCARV | `df.merge()` |
| Calculos | Formula celda | Operacion vectorizada |
| Limite filas | ~1 millon | Sin limite practico |
| Reproducible | No | Si (es codigo) |

In [6]:
# Importar pandas — siempre con el alias pd
import pandas as pd

print(f'Pandas version: {pd.__version__}')
print('Pandas importado correctamente')

Pandas version: 2.3.1
Pandas importado correctamente


---
<a id='6-series'></a>
### 6. Series — El Array Etiquetado

Una **Serie** es un array unidimensional con etiquetas (index).
Es como una columna de Excel con nombre en cada fila.

Tiene dos partes: el **index** (etiquetas) y los **values** (datos).

In [7]:
import pandas as pd

# ── Crear Series ─────────────────────────────────────────────────────────────

# Desde lista — index automatico (0, 1, 2...)
primas = pd.Series([2400, 6200, 1900, 14500, 3200])
print('Desde lista:')
print(primas)
print(f'Tipo: {type(primas)}')

Desde lista:
0     2400
1     6200
2     1900
3    14500
4     3200
dtype: int64
Tipo: <class 'pandas.core.series.Series'>


In [ ]:
# Desde lista con index personalizado
primas_named = pd.Series(
    [2400, 6200, 1900, 14500, 3200],
    index=['P01', 'P02', 'P03', 'P04', 'P05'],
    name='prima_anual'
)
print(primas_named)
print()

# Acceder por etiqueta
print('Prima P03:', primas_named['P03'])

# Acceder por posicion
print('Primera: ', primas_named.iloc[0])
print('Ultima:  ', primas_named.iloc[-1])

P01     2400
P02     6200
P03     1900
P04    14500
P05     3200
Name: prima_anual, dtype: int64

Prima P03: 1900
Primera:  2400
Ultima:   3200


In [8]:
# Desde diccionario — las llaves se vuelven el index
primas_dict = pd.Series({
    'Ana Torres':   2400,
    'Luis Perez':   6200,
    'Maria Garcia': 1900,
    'Carlos Ruiz':  14500,
})
print(primas_dict)

Ana Torres       2400
Luis Perez       6200
Maria Garcia     1900
Carlos Ruiz     14500
dtype: int64


In [9]:
# ── Operaciones sobre Series ─────────────────────────────────────────────────
primas = pd.Series([2400, 6200, 1900, 14500, 3200])

# Estadisticas
print(f'Total:   ${primas.sum():>10,.2f}')
print(f'Promedio:${primas.mean():>10,.2f}')
print(f'Maximo:  ${primas.max():>10,.2f}')
print(f'Minimo:  ${primas.min():>10,.2f}')
print(f'Mediana: ${primas.median():>10,.2f}')
print(f'Desv std:${primas.std():>10,.2f}')
print()

# Resumen completo
print(primas.describe())

Total:   $ 28,200.00
Promedio:$  5,640.00
Maximo:  $ 14,500.00
Minimo:  $  1,900.00
Mediana: $  3,200.00
Desv std:$  5,226.18

count        5.000000
mean      5640.000000
std       5226.184076
min       1900.000000
25%       2400.000000
50%       3200.000000
75%       6200.000000
max      14500.000000
dtype: float64


In [15]:
# ── Operaciones vectorizadas — sin for! ──────────────────────────────────────
# Pandas aplica la operacion a TODOS los elementos a la vez
primas = pd.Series([2400, 6200, 1900, 14500, 3200])

print('Con IVA (x1.16):')
print(primas * 1.16)

print('Prima mensual (/12):')
print((primas / 12).round(2))

print('Mayores de $5,000:')
print(primas[primas > 5000])

# Contar cuantas cumplen la condicion
print(f'Primas > $5,000: {(primas > 5000).sum()}')

Con IVA (x1.16):
0     2784.0
1     7192.0
2     2204.0
3    16820.0
4     3712.0
dtype: float64
Prima mensual (/12):
0     200.00
1     516.67
2     158.33
3    1208.33
4     266.67
dtype: float64
Mayores de $5,000:
1     6200
3    14500
dtype: int64
Primas > $5,000: 2


---
<a id='7-dataframe'></a>
### 7. DataFrame — La Tabla de Pandas

Un **DataFrame** es una tabla bidimensional — una coleccion de Series
que comparten el mismo index. Es el equivalente de una hoja de Excel.

Cada **columna** es una Serie. Cada **fila** es un registro.

In [10]:
import pandas as pd

# ── Crear DataFrame desde diccionario de listas ───────────────────────────────
# Forma mas comun — cada clave es una columna
cartera = pd.DataFrame({
    'poliza':    ['P01', 'P02', 'P03', 'P04', 'P05'],
    'titular':   ['Ana', 'Luis', 'Maria', 'Carlos', 'Rosa'],
    'edad':      [28,   45,    32,     62,     38],
    'prima':     [2400, 6200, 1900, 14500,  3200],
    'siniest':   [0,    1,    0,      3,      0],
    'activa':    [True, True, True,  True,  False],
})

# En Jupyter: cartera  ← muestra tabla visual
# En script:  print(cartera)
cartera

,poliza,titular,edad,prima,siniest,activa
0,P01,Ana,28,2400,0,True
1,P02,Luis,45,6200,1,True
2,P03,Maria,32,1900,0,True
3,P04,Carlos,62,14500,3,True
4,P05,Rosa,38,3200,0,False


In [11]:
# ── Exploracion inicial — lo primero que haces con cualquier dataset ──────────

print('FORMA (filas, columnas):', cartera.shape)
print('FILAS:', len(cartera))
print()

print('PRIMERAS 3 FILAS:')
print(cartera.head(3))
print()

print('ULTIMAS 2 FILAS:')
print(cartera.tail(2))

FORMA (filas, columnas): (5, 6)
FILAS: 5

PRIMERAS 3 FILAS:
  poliza titular  edad  prima  siniest  activa
0    P01     Ana    28   2400        0    True
1    P02    Luis    45   6200        1    True
2    P03   Maria    32   1900        0    True

ULTIMAS 2 FILAS:
  poliza titular  edad  prima  siniest  activa
3    P04  Carlos    62  14500        3    True
4    P05    Rosa    38   3200        0   False


In [12]:
# .info() — tipos de datos y valores nulos
cartera.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   poliza   5 non-null      object
 1   titular  5 non-null      object
 2   edad     5 non-null      int64 
 3   prima    5 non-null      int64 
 4   siniest  5 non-null      int64 
 5   activa   5 non-null      bool  
dtypes: bool(1), int64(3), object(2)
memory usage: 337.0+ bytes


In [13]:
# .describe() — estadisticas descriptivas de columnas numericas
cartera.describe()

,edad,prima,siniest
count,5.000000,5.000000,5.00000
mean,41.000000,5640.000000,0.80000
std,13.379088,5226.184076,1.30384
min,28.000000,1900.000000,0.00000
25%,32.000000,2400.000000,0.00000
50%,38.000000,3200.000000,0.00000
75%,45.000000,6200.000000,1.00000
max,62.000000,14500.000000,3.00000


---
<a id='8-acceso'></a>
### 8. Acceder, Filtrar y Crear Columnas


In [20]:
# ── Acceder a columnas ───────────────────────────────────────────────────────

# Una columna → devuelve Serie
print(cartera['prima'])
print()

# Multiples columnas → devuelve DataFrame
print(cartera[['titular', 'prima', 'activa']])

0     2400
1     6200
2     1900
3    14500
4     3200
Name: prima, dtype: int64

  titular  prima  activa
0     Ana   2400    True
1    Luis   6200    True
2   Maria   1900    True
3  Carlos  14500    True
4    Rosa   3200   False


In [21]:
# ── Acceder a filas ──────────────────────────────────────────────────────────

# Por posicion con .iloc[]
print('Fila 0 (primera):')
print(cartera.iloc[0])
print()

print('Filas 1 a 3:')
print(cartera.iloc[1:4])
print()

# Celda especifica: fila 2, columna 3
print('Prima de la fila 2:', cartera.iloc[2, 3])

# Por etiqueta con .loc[]
print('Prima de P01:', cartera.loc[0, 'prima'])

Fila 0 (primera):
poliza      P01
titular     Ana
edad         28
prima      2400
siniest       0
activa     True
Name: 0, dtype: object

Filas 1 a 3:
  poliza titular  edad  prima  siniest  activa
1    P02    Luis    45   6200        1    True
2    P03   Maria    32   1900        0    True
3    P04  Carlos    62  14500        3    True

Prima de la fila 2: 1900
Prima de P01: 2400


In [22]:
# ── Filtrar filas (Boolean Indexing) ─────────────────────────────────────────
# La forma mas importante de pandas — la usaras constantemente

# Paso 1: crear mascara booleana
mascara = cartera['prima'] > 5000
print('Mascara:')
print(mascara)
print()

# Paso 2: aplicar la mascara
print('Polizas con prima > $5,000:')
print(cartera[mascara])

Mascara:
0    False
1     True
2    False
3     True
4    False
Name: prima, dtype: bool

Polizas con prima > $5,000:
  poliza titular  edad  prima  siniest  activa
1    P02    Luis    45   6200        1    True
3    P04  Carlos    62  14500        3    True


In [ ]:
# ── Filtros combinados ───────────────────────────────────────────────────────

# AND con &  (ambas condiciones deben cumplirse)
filtro_and = cartera[
    (cartera['prima'] > 5000) & (cartera['activa'] == True)
]
print('Prima > 5000 Y activa:')
print(filtro_and[['titular', 'prima', 'activa']])
print()

# OR con |  (al menos una condicion)
filtro_or = cartera[
    (cartera['edad'] < 30) | (cartera['siniest'] > 0)
]
print('Edad < 30 O con siniestros:')
print(filtro_or[['titular', 'edad', 'siniest']])
print()

# .query() — forma mas legible para filtros simples
print(cartera.query('prima > 5000 and activa == True'))

print(cartera.query('edad < 30 or siniest > 0'))


Prima > 5000 Y activa:
  titular  prima  activa
1    Luis   6200    True
3  Carlos  14500    True

Edad < 30 O con siniestros:
  titular  edad  siniest
0     Ana    28        0
1    Luis    45        1
3  Carlos    62        3

  poliza titular  edad  prima  siniest  activa
1    P02    Luis    45   6200        1    True
3    P04  Carlos    62  14500        3    True
  poliza titular  edad  prima  siniest  activa
0    P01     Ana    28   2400        0    True
1    P02    Luis    45   6200        1    True
3    P04  Carlos    62  14500        3    True


In [26]:
# ── Agregar columnas derivadas ────────────────────────────────────────────────

# Columna numerica calculada
cartera['prima_mensual'] = cartera['prima'] / 12

# Columna con pd.cut — rangos de edad
cartera['grupo_edad'] = pd.cut(
    cartera['edad'],
    bins=[0, 30, 45, 60, 100],
    labels=['18-30', '31-45', '46-60', '61+']
)

# Columna de texto con condicion
cartera['nivel_riesgo'] = 'BAJO'
cartera.loc[cartera['siniest'] >= 1, 'nivel_riesgo'] = 'MEDIO'
cartera.loc[cartera['siniest'] >= 3, 'nivel_riesgo'] = 'ALTO'

# Ver resultado
# cartera[['titular', 'prima', 'prima_mensual', 'grupo_edad', 'nivel_riesgo']]
cartera

,poliza,titular,edad,prima,siniest,activa,prima_mensual,grupo_edad,nivel_riesgo
0,P01,Ana,28,2400,0,True,200.000000,18-30,BAJO
1,P02,Luis,45,6200,1,True,516.666667,31-45,MEDIO
2,P03,Maria,32,1900,0,True,158.333333,31-45,BAJO
3,P04,Carlos,62,14500,3,True,1208.333333,61+,ALTO
4,P05,Rosa,38,3200,0,False,266.666667,31-45,BAJO


---
<a id='9-ejercicio'></a>
## 9. Ejercicio Integrador — Modulos + Pandas

Combinamos `mi_modulo.py` con Pandas para analizar una cartera completa.

In [25]:
# Si modificas mi_modulo.py y quieres recargar sin reiniciar el kernel:
import importlib
import mi_modulo
importlib.reload(mi_modulo)
print('Modulo recargado')

Modulo recargado


In [1]:
# ── Datos de la cartera ──────────────────────────────────────────────────────
import pandas as pd
from mi_modulo import calcular_prima, clasificar_riesgo, grupo_edad

cartera = pd.DataFrame({
    'poliza':   ['A-01', 'A-02', 'A-03', 'A-04', 'A-05'],
    'nombre':   ['Sofia R.','Miguel T.','Laura G.','Roberto M.','Carmen P.'],
    'edad':     [24, 38, 31, 55, 42],
    'vehiculo': ['Sedan', 'SUV', 'Hatchback', 'SUV', 'Sedan'],
    'siniest':  [0, 1, 0, 2, 0],
    'suma':     [300_000, 500_000, 250_000, 600_000, 350_000],
})

tarifas_vehiculo = {
    'Sedan':    0.025,
    'SUV':      0.035,
    'Hatchback':0.022,
    'Pick-up':  0.040,
}

print(f'Cartera cargada: {len(cartera)} polizas')

Cartera cargada: 5 polizas


In [2]:
# ── Tarea 1: Calcular primas usando mi_modulo ────────────────────────────────
# Crea la columna 'prima' usando calcular_prima() de mi_modulo
# Para cada poliza: suma * tarifas_vehiculo[vehiculo]
import mi_modulo
# Tu codigo aqui:
cartera['prima'] = cartera.apply(
    lambda f: mi_modulo.calcular_prima_auto(f['suma'], f['vehiculo']),
    axis=1
)

In [27]:
cartera

,poliza,nombre,edad,vehiculo,siniest,suma,prima
0,A-01,Sofia R.,24,Sedan,0,300000,7500.0
1,A-02,Miguel T.,38,SUV,1,500000,17500.0
2,A-03,Laura G.,31,Hatchback,0,250000,5500.0
3,A-04,Roberto M.,55,SUV,2,600000,21000.0
4,A-05,Carmen P.,42,Sedan,0,350000,8750.0


In [3]:
# ── Tarea 2: Columnas derivadas con mi_modulo ────────────────────────────────
# Crea 'riesgo' usando clasificar_riesgo(siniest) de mi_modulo
# Crea 'grupo_edad' usando grupo_edad(edad) de mi_modulo

# Tu codigo aqui:
cartera['riesgo'] = cartera.apply(lambda f: mi_modulo.clasificar_riesgo(f['siniest']), axis = 1)
cartera['grupo_edad'] = cartera.apply(lambda f: mi_modulo.grupo_edad(f['edad']), axis=1)
cartera

,poliza,nombre,edad,vehiculo,siniest,suma,prima,riesgo,grupo_edad
0,A-01,Sofia R.,24,Sedan,0,300000,7500.0,BAJO,18-30
1,A-02,Miguel T.,38,SUV,1,500000,17500.0,MEDIO,31-45
2,A-03,Laura G.,31,Hatchback,0,250000,5500.0,BAJO,31-45
3,A-04,Roberto M.,55,SUV,2,600000,21000.0,MEDIO,46-60
4,A-05,Carmen P.,42,Sedan,0,350000,8750.0,BAJO,31-45


In [4]:
# ── Tarea 3: Analisis con pandas ─────────────────────────────────────────────
# 3a. Muestra solo las polizas de riesgo MEDIO o ALTO
# 3b. Calcula prima total y promedio de toda la cartera
# 3c. Muestra la poliza con la prima mas alta (usa .idxmax())

# Tu codigo aqui:
a3 = (cartera['riesgo'] == 'BAJO') | (cartera['riesgo']=='MEDIO')
print(cartera[a3])

  poliza      nombre  edad   vehiculo  siniest    suma    prima riesgo  \
0   A-01    Sofia R.    24      Sedan        0  300000   7500.0   BAJO   
1   A-02   Miguel T.    38        SUV        1  500000  17500.0  MEDIO   
2   A-03    Laura G.    31  Hatchback        0  250000   5500.0   BAJO   
3   A-04  Roberto M.    55        SUV        2  600000  21000.0  MEDIO   
4   A-05   Carmen P.    42      Sedan        0  350000   8750.0   BAJO   

  grupo_edad  
0      18-30  
1      31-45  
2      31-45  
3      46-60  
4      31-45  


In [7]:
print(f"Prima total: {cartera['prima'].sum()}")
print(f"Prima promedio: {cartera['prima'].mean()}")

Prima total: 60250.0
Prima promedio: 12050.0


In [6]:
print(f"Poliza con la prima más alta: {cartera.loc[cartera['prima'].idxmax(), 'poliza']}")

Poliza con la prima más alta: A-04


---
## Resumen de la Sesion 6

### T4 — COMPLETO
| Concepto | Lo que aprendimos |
|---------|------------------|
| **Modulos propios** | Crear .py con funciones, importar con `import` y `from` |
| **tabla_tarifas.py** | Clase actuarial real — set_tarifa(), get_tarifa(), __repr__ |
| **Scripts .py** | `if __name__ == '__main__'` — ejecutar desde terminal |

### T5 — Inicio
| Concepto | Lo que aprendimos |
|---------|------------------|
| **Series** | Array unidimensional con index, operaciones vectorizadas |
| **DataFrame** | Tabla bidimensional, `.head()`, `.info()`, `.describe()` |
| **Acceso** | `df['col']`, `.iloc[]`, `.loc[]`, celda individual |
| **Filtros** | Boolean indexing, `&`, `|`, `.query()` |
| **Columnas** | Derivadas por operacion, `pd.cut()`, condicion con `.loc[]` |

**Proxima sesion — Mie 29 de abril, 18:00-21:00 h:**
T5 avanzado: merge, groupby, valores nulos, lectura de CSV/Excel/Parquet.

**Tarea:**
```bash
git add sesion6_M1_notebook.ipynb mi_modulo.py
git commit -m "Sesion 6: modulos propios + pandas Series y DataFrame"
git push
```

---
*Diplomado ML en Seguros · Facultad de Ciencias, UNAM · 2026*